#  Tomato Disease Detection
### Using Convolutional Autoencoder (CAE) + CNN Classifier

**What this notebook does:**
1. Downloads the PlantVillage dataset from Kaggle
2. Trains a Convolutional Autoencoder (CAE)
3. Trains a CNN classifier on CAE output
4. Predicts disease and suggests treatment

**Before running:** Go to `Runtime` → `Change runtime type` → Select **GPU**

## Step 1: Connect Google Drive (to save your models)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive connected!')

Mounted at /content/drive
✅ Google Drive connected!


## Step 2: Set up Kaggle API

1. Go to [https://www.kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to **API** section
3. Under Legacy API Credentials click **Create Legacy API Key** — this downloads `kaggle.json`
4. Run the cell below and upload that file when prompted

In [2]:
from google.colab import files
import os

# Upload your kaggle.json
print('Upload your kaggle.json file:')
uploaded = files.upload()

# Place it where Kaggle expects it
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('✅ Kaggle API key set up successfully!')

Upload your kaggle.json file:


Saving kaggle.json to kaggle.json
✅ Kaggle API key set up successfully!


## Step 3: Download the PlantVillage Dataset

In [3]:
!pip install kaggle --quiet
!kaggle datasets download -d abdallahalidev/plantvillage-dataset --unzip -p ./plantvillage
print('✅ Dataset downloaded and unzipped!')

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:20<00:00, 107MB/s]

✅ Dataset downloaded and unzipped!


## Step 4: Imports & Config

In [4]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D,
    Reshape, UpSampling2D, Flatten, Dropout
)
from tensorflow.keras.models import Model

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Image config
H, W, C = 128, 128, 3

# Dataset path (adjust if folder structure differs)
TRAIN_DIR = './plantvillage/plantvillage dataset/color'

CLASS_NAMES = [
    'Tomato_Bacterial_spot',
    'Tomato_Early_blight',
    'Tomato_Late_blight',
    'Tomato_Leaf_Mold',
    'Tomato_Septoria_leaf_spot',
    'Tomato_Spider_mites_Two_spotted_spider_mite',
    'Tomato__Target_Spot',
    'Tomato__Tomato_YellowLeaf__Curl_Virus',
    'Tomato__Tomato_mosaic_virus',
    'Tomato_healthy'
]

SOLUTIONS = {
    0: "Plow down crop residue after harvest. Use fixed-copper based products for managing bacterial spot.",
    1: "Crush 1 aspirin into powder, mix with 4 cups of water. Spray every 2-3 weeks during the growing season.",
    2: "Mix baking soda, vegetable oil, and mild soap. Spray on plants and reapply regularly.",
    3: "Apple-cider and vinegar mix treats mold effectively. Corn and garlic spray prevents fungi outbreaks.",
    4: "Use raised beds, rotate crops, and mulch. Apply fungicides with chlorothalonil or copper before disease appears.",
    5: "Use Bifenazate (Acramite) at 0.75-1 lb/acre. Effective against spider mites with low toxicity to beneficial insects.",
    6: "Remove old plant debris. Rotate crops. Ensure good air circulation and water plants in the morning.",
    7: "No cure for TYLCV. Remove and destroy infected plants immediately to prevent spread.",
    8: "No cure for Mosaic Virus. Remove infected plants. Avoid planting in soil with root debris.",
    9: "Your tomato plant is healthy! Keep up the good care. 🌱"
}

print('✅ Imports and config done!')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

✅ Imports and config done!
TensorFlow version: 2.20.0
GPU available: True


## Step 5: Load & Preprocess Images

In [5]:
# Check what folders are available
print('Folders found in dataset:')
for folder in sorted(os.listdir(TRAIN_DIR)):
    if 'Tomato' in folder:
        print(f'  📁 {folder}')

Folders found in dataset:
  📁 Tomato___Bacterial_spot
  📁 Tomato___Early_blight
  📁 Tomato___Late_blight
  📁 Tomato___Leaf_Mold
  📁 Tomato___Septoria_leaf_spot
  📁 Tomato___Spider_mites Two-spotted_spider_mite
  📁 Tomato___Target_Spot
  📁 Tomato___Tomato_Yellow_Leaf_Curl_Virus
  📁 Tomato___Tomato_mosaic_virus
  📁 Tomato___healthy


In [8]:
data = []
labels = []
print(TRAIN_DIR)
for label_idx, class_name in enumerate(CLASS_NAMES):
    pattern = os.path.join(TRAIN_DIR, f'{class_name}')
    print(pattern)
    image_paths = glob.glob(pattern)

    if not image_paths:
        print(f'No images found for: {class_name}')
        continue

    print(f'Loading {len(image_paths)} images for {class_name}...')

    for img_path in image_paths:
        try:
            image = tf.keras.preprocessing.image.load_img(
                img_path, color_mode='rgb', target_size=(H, W)
            )
            image = np.array(image, dtype='float32') / 255.0
            data.append(image)
            labels.append(label_idx)
        except Exception as e:
            print(f'  Skipping {img_path}: {e}')

data = np.array(data)
labels = np.array(labels)

print(f'\nTotal images loaded: {len(data)}')
print(f'Data shape: {data.shape}')

./plantvillage/plantvillage dataset/color
./plantvillage/plantvillage dataset/color/Tomato_Bacterial_spot
No images found for: Tomato_Bacterial_spot
./plantvillage/plantvillage dataset/color/Tomato_Early_blight
No images found for: Tomato_Early_blight
./plantvillage/plantvillage dataset/color/Tomato_Late_blight
No images found for: Tomato_Late_blight
./plantvillage/plantvillage dataset/color/Tomato_Leaf_Mold
No images found for: Tomato_Leaf_Mold
./plantvillage/plantvillage dataset/color/Tomato_Septoria_leaf_spot
No images found for: Tomato_Septoria_leaf_spot
./plantvillage/plantvillage dataset/color/Tomato_Spider_mites_Two_spotted_spider_mite
No images found for: Tomato_Spider_mites_Two_spotted_spider_mite
./plantvillage/plantvillage dataset/color/Tomato__Target_Spot
No images found for: Tomato__Target_Spot
./plantvillage/plantvillage dataset/color/Tomato__Tomato_YellowLeaf__Curl_Virus
No images found for: Tomato__Tomato_YellowLeaf__Curl_Virus
./plantvillage/plantvillage dataset/color/

## Step 6: Train/Test Split & Visualise Samples

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.3, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')

In [ ]:
plt.figure(figsize=(15, 15))
for i in range(10):
    ax = plt.subplot(5, 2, i + 1)
    plt.imshow(X_train[i])
    plt.title(CLASS_NAMES[y_train[i]], fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Step 7: Build & Train the Convolutional Autoencoder (CAE)

In [ ]:
inputs = Input(shape=(H, W, C))

# ── Encoder ──
x = Conv2D(128, (3,3), activation='relu', padding='same')(inputs)
x = MaxPooling2D((2,2))(x)
x = Conv2D(64,  (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(64,  (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(32,  (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(32,  (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(16,  (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)

# Save shape for decoder reshape
encoder_shape = x.shape[1:]  # e.g. (2, 2, 16)
flat_size = encoder_shape[0] * encoder_shape[1] * encoder_shape[2]
print(f'Encoder output shape: {encoder_shape}, flat size: {flat_size}')

# ── Bottleneck ──
x = Flatten()(x)
encoded = Dense(10, activation='relu')(x)
encoded = Dropout(0.2)(encoded)

# ── Decoder ──
d = Dense(flat_size, activation='relu')(encoded)
d = Dropout(0.2)(d)
d = Reshape(encoder_shape)(d)

d = UpSampling2D((2,2))(d)
d = Conv2D(16,  (3,3), activation='relu', padding='same')(d)
d = UpSampling2D((2,2))(d)
d = Conv2D(32,  (3,3), activation='relu', padding='same')(d)
d = UpSampling2D((2,2))(d)
d = Conv2D(32,  (3,3), activation='relu', padding='same')(d)
d = UpSampling2D((2,2))(d)
d = Conv2D(64,  (3,3), activation='relu', padding='same')(d)
d = UpSampling2D((2,2))(d)
d = Conv2D(128, (3,3), activation='relu', padding='same')(d)
d = UpSampling2D((2,2))(d)
d = Conv2D(3,   (3,3), activation='sigmoid', padding='same')(d)

autoencoder = Model(inputs, d, name='autoencoder')
autoencoder.summary()

In [ ]:
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(
    X_train, X_train,
    batch_size=16,
    epochs=3,
    validation_data=(X_test, X_test)
)

# Save to Google Drive
os.makedirs('/content/drive/MyDrive/TomatoModel', exist_ok=True)
autoencoder.save('/content/drive/MyDrive/TomatoModel/cae.h5')
print('✅ CAE saved to Google Drive!')

## Step 8: Generate Features Using Trained CAE

In [ ]:
X_train_cae = autoencoder.predict(X_train, verbose=1)
X_test_cae  = autoencoder.predict(X_test,  verbose=1)
print(f'✅ CAE output shape: {X_train_cae.shape}')

## Step 9: Build & Train the CNN Classifier

In [ ]:
inputs2 = Input(shape=(H, W, C))

e = Conv2D(128, (3,3), activation='relu', padding='same')(inputs2)
e = MaxPooling2D((2,2))(e)
e = Conv2D(64,  (3,3), activation='relu', padding='same')(e)
e = MaxPooling2D((2,2))(e)
e = Conv2D(64,  (3,3), activation='relu', padding='same')(e)
e = MaxPooling2D((2,2))(e)
e = Conv2D(32,  (3,3), activation='relu', padding='same')(e)
e = MaxPooling2D((2,2))(e)
e = Conv2D(32,  (3,3), activation='relu', padding='same')(e)
e = MaxPooling2D((2,2))(e)
e = Conv2D(16,  (3,3), activation='relu', padding='same')(e)
e = MaxPooling2D((2,2))(e)

e = Flatten()(e)
e = Dense(10,   activation='relu')(e)
e = Dropout(0.2)(e)
e = Dense(1000, activation='relu')(e)
e = Dropout(0.3)(e)
e = Dense(700,  activation='relu')(e)
e = Dropout(0.5)(e)
e = Dense(350,  activation='relu')(e)
output = Dense(10, activation='softmax')(e)

cnn_model = Model(inputs2, output, name='cnn_classifier')
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = cnn_model.fit(
    X_train_cae, y_train,
    batch_size=32,
    epochs=5,
    validation_split=0.2
)

cnn_model.save('/content/drive/MyDrive/TomatoModel/cnn.h5')
print('✅ CNN saved to Google Drive!')

## Step 10: Evaluate & Plot Results

In [ ]:
print('\n📊 Evaluation on test set:')
cnn_model.evaluate(X_test_cae, y_test, verbose=2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

## Step 11: Predict Disease & Show Solution

In [ ]:
y_pred = np.argmax(cnn_model.predict(X_test_cae), axis=1)

# Show result for first 5 test images
plt.figure(figsize=(15, 6))
for i in range(5):
    ax = plt.subplot(1, 5, i + 1)
    plt.imshow(X_test[i])
    predicted = CLASS_NAMES[y_pred[i]]
    actual    = CLASS_NAMES[y_test[i]]
    color     = 'green' if y_pred[i] == y_test[i] else 'red'
    plt.title(f'Pred: {predicted.split("_")[1]}\nActual: {actual.split("_")[1]}',
              color=color, fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

# Print solution for first prediction
print(f'\n🍅 Disease  : {CLASS_NAMES[y_pred[0]]}')
print(f'💊 Solution : {SOLUTIONS[y_pred[0]]}')